In [28]:
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
from nemosis import static_table, dynamic_data_compiler, defaults
import plotly.express as px
import os
import glob
import dask.dataframe as dd

pd.set_option('display.max_columns', None)

In [32]:
import glob
import os

file_list = glob.glob('/Volumes/T7/bid-volume-data-sorted/*.feather')

# Sort alphabetically by filename
file_list.sort()

if not file_list:
    print("No files found.")
else:
    print("First entry:", file_list[0])
    print("Last entry:", file_list[-1])

First entry: /Volumes/T7/bid-volume-data-sorted/PUBLIC_DVD_BIDPEROFFER_D_20090701.feather
Last entry: /Volumes/T7/bid-volume-data-sorted/PUBLIC_DVD_BIDPEROFFER_D_20241112.feather


In [33]:
# Filter feather files for RaiseReg and LowerReg BIDTYPEs and save as parquet files
# Create the output directory if it doesn't already exist
output_dir = '/Volumes/T7/bid-volume-filtered-2'
os.makedirs(output_dir, exist_ok=True)

file_list = glob.glob('/Volumes/T7/bid-volume-data-sorted/*.feather')

# Sort alphabetically by filename
file_list.sort()

for file in file_list:
    print(f"Processing {file}...")
    # Read the Feather file
    df = pd.read_feather(file)
    
    # Filter to keep rows where BIDTYPE is either "RAISEREG" or "LOWERREG"
    filtered_volume_bids = df[(df["BIDTYPE"] == "RAISEREG") | (df["BIDTYPE"] == "LOWERREG")]

    # Print the first few rows of the filtered DataFrame
    print("Filtered DataFrame head:")
    print(filtered_volume_bids.head())

    # Get just the filename without the path
    base_filename = os.path.basename(file)  # e.g. "PUBLIC_DVD_BIDPEROFFER_D_201201010000.feather"
    # Remove '.feather' extension
    filename_no_ext = os.path.splitext(base_filename)[0]  # e.g. "PUBLIC_DVD_BIDPEROFFER_D_201201010000"

    # Construct the full output path in output_dir
    out_file = os.path.join(output_dir, filename_no_ext + ".parquet")

    # Save to Parquet
    filtered_volume_bids.to_parquet(out_file, index=False)
    print(f"Finished processing {file} -> {out_file}\n")

Processing /Volumes/T7/bid-volume-data-sorted/PUBLIC_DVD_BIDPEROFFER_D_20090701.feather...
Filtered DataFrame head:
         SETTLEMENTDATE      DUID   BIDTYPE            OFFERDATE MAXAVAIL  \
25  2009/07/01 00:00:00   BASTYAN  LOWERREG  2009/06/30 12:10:15       26   
29  2009/07/01 00:00:00   BASTYAN  RAISEREG  2009/06/30 12:10:15       26   
39  2009/07/01 00:00:00  BELLBAY1  LOWERREG  2005/04/21 14:31:20        0   
42  2009/07/01 00:00:00  BELLBAY1  RAISEREG  2005/04/21 14:31:20        0   
47  2009/07/01 00:00:00  BELLBAY2  LOWERREG  2005/04/21 14:31:20        0   

   ENABLEMENTMIN ENABLEMENTMAX LOWBREAKPOINT HIGHBREAKPOINT BANDAVAIL1  \
25            25            78            51             78          0   
29             0            78             0             52          0   
39            36           120            49            120          3   
42            36           115            36            115          3   
47            36           120            49       

In [36]:
import pandas as pd

def inspect_parquet_file(file_path):
    """
    Load a parquet file and display its structure and first few rows
    
    Parameters:
    -----------
    file_path : str
        Path to the parquet file
    """
    print(f"Loading parquet file: {file_path}")
    
    # Load the parquet file
    df = pd.read_parquet(file_path)
    
    # Display basic information
    print("\n==== Basic Information ====")
    print(f"Number of rows: {len(df)}")
    print(f"Number of columns: {len(df.columns)}")
    
    # Display column names and types
    print("\n==== Column Information ====")
    for col, dtype in zip(df.columns, df.dtypes):
        print(f"{col}: {dtype}")
    
    # Check if SETTLEMENTDATE exists
    if 'SETTLEMENTDATE' in df.columns:
        print("\n==== SETTLEMENTDATE Information ====")
        settlement_dtype = df['SETTLEMENTDATE'].dtype
        print(f"SETTLEMENTDATE data type: {settlement_dtype}")
        
        # Show sample values
        unique_dates = df['SETTLEMENTDATE'].unique()
        num_unique = len(unique_dates)
        print(f"Number of unique settlement dates: {num_unique}")
        print(f"Sample of settlement dates (first 5):")
        for i, date in enumerate(unique_dates[:5]):
            print(f"  {i+1}. {date}")
            
        # Try to extract time components if datetime
        if pd.api.types.is_datetime64_any_dtype(df['SETTLEMENTDATE']):
            try:
                # Count occurrences of 18:05:00
                count_1805 = df[
                    (df['SETTLEMENTDATE'].dt.hour == 18) & 
                    (df['SETTLEMENTDATE'].dt.minute == 5) & 
                    (df['SETTLEMENTDATE'].dt.second == 0)
                ].shape[0]
                print(f"\nRows with settlement time 18:05:00: {count_1805}")
            except Exception as e:
                print(f"Error extracting time components: {str(e)}")
        else:
            # Try string-based approach
            try:
                count_1805 = df[df['SETTLEMENTDATE'].astype(str).str.contains('18:05:00')].shape[0]
                print(f"\nRows with settlement time 18:05:00: {count_1805}")
            except Exception as e:
                print(f"Error checking for 18:05:00 via string method: {str(e)}")
    
    # Display the first few rows
    print("\n==== First 5 Rows ====")
    print(df.head())
    
    return df

if __name__ == "__main__":
    # File path
    file_path = "/Volumes/T7/bid-volume-filtered-2/PUBLIC_DVD_BIDPEROFFER_D_20241013.parquet"
    
    # Inspect the file
    df = inspect_parquet_file(file_path)

Loading parquet file: /Volumes/T7/bid-volume-filtered-2/PUBLIC_DVD_BIDPEROFFER_D_20241013.parquet

==== Basic Information ====
Number of rows: 86400
Number of columns: 21

==== Column Information ====
SETTLEMENTDATE: object
DUID: object
BIDTYPE: object
OFFERDATE: object
MAXAVAIL: object
ENABLEMENTMIN: object
ENABLEMENTMAX: object
LOWBREAKPOINT: object
HIGHBREAKPOINT: object
BANDAVAIL1: object
BANDAVAIL2: object
BANDAVAIL3: object
BANDAVAIL4: object
BANDAVAIL5: object
BANDAVAIL6: object
BANDAVAIL7: object
BANDAVAIL8: object
BANDAVAIL9: object
BANDAVAIL10: object
INTERVAL_DATETIME: object
DIRECTION: object

==== SETTLEMENTDATE Information ====
SETTLEMENTDATE data type: object
Number of unique settlement dates: 1
Sample of settlement dates (first 5):
  1. 2024/10/13 00:00:00

Rows with settlement time 18:05:00: 0

==== First 5 Rows ====
        SETTLEMENTDATE    DUID   BIDTYPE            OFFERDATE MAXAVAIL  \
0  2024/10/13 00:00:00  ADPBA1  LOWERREG  2024/10/13 03:58:05        0   
1  202

In [38]:
import os
import glob
import pandas as pd
import re

def filter_interval_time(input_dir, output_dir, target_time='18:05:00'):
    """
    Filter parquet files for rows where INTERVAL_DATETIME contains a specific time (e.g., '18:05:00')
    
    Parameters:
    -----------
    input_dir : str
        Directory containing the parquet files to filter
    output_dir : str
        Directory where filtered files will be saved
    target_time : str
        Time to filter for in format 'HH:MM:SS'
    """
    print(f"Starting to filter data for interval time: {target_time}")
    
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output directory created/verified: {output_dir}")
    
    # Get list of all parquet files in the input directory
    file_list = glob.glob(os.path.join(input_dir, "*.parquet"))
    
    # Sort the file list to ensure consistent processing order
    file_list.sort()
    
    print(f"Found {len(file_list)} parquet files to process")
    
    # Counter for tracking progress
    total_files = len(file_list)
    files_processed = 0
    rows_filtered = 0
    total_rows_processed = 0
    files_with_target = 0
    
    for file_path in file_list:
        file_name = os.path.basename(file_path)
        files_processed += 1
        
        print(f"Processing file {files_processed}/{total_files}: {file_name}")
        
        try:
            # Read the parquet file
            df = pd.read_parquet(file_path)
            
            # Track total rows for statistics
            file_row_count = len(df)
            total_rows_processed += file_row_count
            
            # Filter for rows where INTERVAL_DATETIME contains the target time
            if 'INTERVAL_DATETIME' in df.columns:
                # Check data type and filter accordingly
                if pd.api.types.is_datetime64_any_dtype(df['INTERVAL_DATETIME']):
                    # If it's a datetime column, extract components
                    hour, minute, second = map(int, target_time.split(':'))
                    mask = (df['INTERVAL_DATETIME'].dt.hour == hour) & \
                           (df['INTERVAL_DATETIME'].dt.minute == minute) & \
                           (df['INTERVAL_DATETIME'].dt.second == second)
                else:
                    # If it's a string column, use string contains method
                    mask = df['INTERVAL_DATETIME'].astype(str).str.contains(target_time)
                
                filtered_df = df[mask]
                
                # Count filtered rows for reporting
                filtered_row_count = len(filtered_df)
                rows_filtered += filtered_row_count
                
                # Only write output if we have rows that match
                if filtered_row_count > 0:
                    files_with_target += 1
                    
                    # Construct output file path
                    output_file_path = os.path.join(output_dir, file_name)
                    
                    # Save the filtered DataFrame to a new parquet file
                    filtered_df.to_parquet(output_file_path, index=False)
                    print(f"  - Saved {filtered_row_count} rows with {target_time} to {output_file_path}")
                else:
                    print(f"  - No rows with {target_time} found in this file")
            else:
                print(f"  - Warning: INTERVAL_DATETIME column not found in {file_name}")
                
        except Exception as e:
            print(f"  - Error processing {file_name}: {str(e)}")
    
    # Print summary statistics
    print("\nProcessing complete!")
    print(f"Processed {total_files} files with {total_rows_processed} total rows")
    print(f"Found {rows_filtered} rows containing interval time {target_time}")
    print(f"Found target time in {files_with_target} out of {total_files} files")
    print(f"Filtered data saved to {output_dir}")

if __name__ == "__main__":
    # Directories
    input_directory = "/Volumes/T7/bid-volume-filtered-2"
    output_directory = "/Volumes/T7/bid-volume-filtered-3"
    
    # Run the filter function for the 18:05:00 time period
    filter_interval_time(
        input_dir=input_directory,
        output_dir=output_directory,
        target_time='18:05:00'
    )

Starting to filter data for interval time: 18:05:00
Output directory created/verified: /Volumes/T7/bid-volume-filtered-3
Found 1448 parquet files to process
Processing file 1/1448: PUBLIC_DVD_BIDPEROFFER_D_20090701.parquet
  - Saved 5828 rows with 18:05:00 to /Volumes/T7/bid-volume-filtered-3/PUBLIC_DVD_BIDPEROFFER_D_20090701.parquet
Processing file 2/1448: PUBLIC_DVD_BIDPEROFFER_D_20090801.parquet
  - Saved 5828 rows with 18:05:00 to /Volumes/T7/bid-volume-filtered-3/PUBLIC_DVD_BIDPEROFFER_D_20090801.parquet
Processing file 3/1448: PUBLIC_DVD_BIDPEROFFER_D_20090901.parquet
  - Saved 5622 rows with 18:05:00 to /Volumes/T7/bid-volume-filtered-3/PUBLIC_DVD_BIDPEROFFER_D_20090901.parquet
Processing file 4/1448: PUBLIC_DVD_BIDPEROFFER_D_20091001.parquet
  - Saved 5766 rows with 18:05:00 to /Volumes/T7/bid-volume-filtered-3/PUBLIC_DVD_BIDPEROFFER_D_20091001.parquet
Processing file 5/1448: PUBLIC_DVD_BIDPEROFFER_D_20091101.parquet
  - Saved 5580 rows with 18:05:00 to /Volumes/T7/bid-volume-f

In [37]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime
import re

def filter_settlement_time(input_dir, output_dir, target_time='18:05:00', examine_first_file=True):
    """
    Filter parquet files for rows where SETTLEMENTDATE contains a specific time (e.g., '18:05:00')
    
    Parameters:
    -----------
    input_dir : str
        Directory containing the parquet files to filter
    output_dir : str
        Directory where filtered files will be saved
    target_time : str
        Time to filter for in format 'HH:MM:SS'
    """
    print(f"Starting to filter data for settlement time: {target_time}")
    
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output directory created/verified: {output_dir}")
    
    # Get list of all parquet files in the input directory
    file_list = glob.glob(os.path.join(input_dir, "*.parquet"))
    
    # Sort the file list to ensure consistent processing order
    file_list.sort()
    
    print(f"Found {len(file_list)} parquet files to process")
    
    # Examine the first file to understand the data structure if requested
    if examine_first_file and file_list:
        try:
            print(f"\nExamining first file to understand data structure...")
            first_file = file_list[0]
            sample_df = pd.read_parquet(first_file)
            
            if 'SETTLEMENTDATE' in sample_df.columns:
                print(f"SETTLEMENTDATE column found.")
                print(f"Data type: {sample_df['SETTLEMENTDATE'].dtype}")
                
                # Display sample values
                sample_values = sample_df['SETTLEMENTDATE'].head(3).tolist()
                print(f"Sample values: {sample_values}")
                
                # Try to parse one value to see its components
                try:
                    if len(sample_df) > 0:
                        first_value = sample_df['SETTLEMENTDATE'].iloc[0]
                        if pd.api.types.is_datetime64_any_dtype(sample_df['SETTLEMENTDATE']):
                            print(f"As datetime components: Year={first_value.year}, Month={first_value.month}, Day={first_value.day}, Hour={first_value.hour}, Minute={first_value.minute}, Second={first_value.second}")
                        else:
                            # Try to convert to datetime
                            parsed = pd.to_datetime(first_value)
                            print(f"Parsed as datetime: {parsed}")
                except Exception as e:
                    print(f"Could not parse datetime components: {str(e)}")
            else:
                print(f"SETTLEMENTDATE column not found. Available columns: {sample_df.columns.tolist()}")
                
            print("Data structure examination complete.\n")
        except Exception as e:
            print(f"Error examining first file: {str(e)}\n")
    
    # Counter for tracking progress
    total_files = len(file_list)
    files_processed = 0
    rows_filtered = 0
    total_rows_processed = 0
    
    for file_path in file_list:
        file_name = os.path.basename(file_path)
        files_processed += 1
        
        print(f"Processing file {files_processed}/{total_files}: {file_name}")
        
        try:
            # Read the parquet file
            df = pd.read_parquet(file_path)
            
            # Track total rows for statistics
            file_row_count = len(df)
            total_rows_processed += file_row_count
            
            # Filter for rows where SETTLEMENTDATE contains the target time
            if 'SETTLEMENTDATE' in df.columns:
                # First, let's examine what we're dealing with
                sample_value = df['SETTLEMENTDATE'].iloc[0] if len(df) > 0 else None
                dtype_name = df['SETTLEMENTDATE'].dtype
                print(f"  - SETTLEMENTDATE column dtype: {dtype_name}")
                print(f"  - Sample value: {sample_value}")
                
                # Check the data type of SETTLEMENTDATE
                if pd.api.types.is_datetime64_any_dtype(df['SETTLEMENTDATE']):
                    # For datetime dtype, extract the time component and compare
                    # This extracts hours, minutes, seconds exactly
                    mask = (df['SETTLEMENTDATE'].dt.hour == 18) & \
                           (df['SETTLEMENTDATE'].dt.minute == 5) & \
                           (df['SETTLEMENTDATE'].dt.second == 0)
                    filtered_df = df[mask]
                    print(f"  - Used datetime filtering method")
                elif 'datetime' in str(dtype_name).lower() or 'timestamp' in str(dtype_name).lower():
                    # For other datetime-like types that might not work with dt accessor
                    try:
                        # Try to convert to pandas datetime first
                        temp_dates = pd.to_datetime(df['SETTLEMENTDATE'])
                        mask = (temp_dates.dt.hour == 18) & \
                               (temp_dates.dt.minute == 5) & \
                               (temp_dates.dt.second == 0)
                        filtered_df = df[mask]
                        print(f"  - Used converted datetime filtering method")
                    except:
                        # Fall back to string comparison if conversion fails
                        mask = df['SETTLEMENTDATE'].astype(str).str.contains(target_time)
                        filtered_df = df[mask]
                        print(f"  - Used string contains method (fallback)")
                else:
                    # For string columns or other types, convert to string
                    mask = df['SETTLEMENTDATE'].astype(str).str.contains(target_time)
                    filtered_df = df[mask]
                    print(f"  - Used string contains method")
                
                # Count filtered rows for reporting
                filtered_row_count = len(filtered_df)
                rows_filtered += filtered_row_count
                
                # Only write output if we have rows that match
                if filtered_row_count > 0:
                    # Construct output file path
                    output_file_path = os.path.join(output_dir, file_name)
                    
                    # Save the filtered DataFrame to a new parquet file
                    filtered_df.to_parquet(output_file_path, index=False)
                    print(f"  - Saved {filtered_row_count} matching rows to {output_file_path}")
                else:
                    print(f"  - No matching rows found in this file")
            else:
                print(f"  - Warning: SETTLEMENTDATE column not found in {file_name}")
                
        except Exception as e:
            print(f"  - Error processing {file_name}: {str(e)}")
    
    # Print summary statistics
    print("\nProcessing complete!")
    print(f"Processed {total_files} files with {total_rows_processed} total rows")
    print(f"Filtered {rows_filtered} rows containing settlement time {target_time}")
    print(f"Filtered data saved to {output_dir}")

if __name__ == "__main__":
    # Directories
    input_directory = "/Volumes/T7/bid-volume-filtered-2"
    output_directory = "/Volumes/T7/bid-volume-filtered-3"
    
    # Extract time components from the target time
    time_parts = "18:05:00".split(":")
    hour = int(time_parts[0])
    minute = int(time_parts[1])
    second = int(time_parts[2])
    
    print(f"Filtering for settlement time {hour}:{minute}:{second}")
    
    # Run the filter function for the 18:05:00 time period
    filter_settlement_time(
        input_dir=input_directory,
        output_dir=output_directory,
        target_time='18:05:00',
        examine_first_file=True  # Set to True to analyze the first file
    )

Filtering for settlement time 18:5:0
Starting to filter data for settlement time: 18:05:00
Output directory created/verified: /Volumes/T7/bid-volume-filtered-3
Found 1448 parquet files to process

Examining first file to understand data structure...
SETTLEMENTDATE column found.
Data type: object
Sample values: ['2009/07/01 00:00:00', '2009/07/01 00:00:00', '2009/07/01 00:00:00']
Parsed as datetime: 2009-07-01 00:00:00
Data structure examination complete.

Processing file 1/1448: PUBLIC_DVD_BIDPEROFFER_D_20090701.parquet
  - SETTLEMENTDATE column dtype: object
  - Sample value: 2009/07/01 00:00:00
  - Used string contains method
  - No matching rows found in this file
Processing file 2/1448: PUBLIC_DVD_BIDPEROFFER_D_20090801.parquet
  - SETTLEMENTDATE column dtype: object
  - Sample value: 2009/08/01 00:00:00
  - Used string contains method
  - No matching rows found in this file
Processing file 3/1448: PUBLIC_DVD_BIDPEROFFER_D_20090901.parquet
  - SETTLEMENTDATE column dtype: object
  

KeyboardInterrupt: 

In [27]:
import os

output_dir = '/Volumes/T7/bid-volume-filtered-2'
os.makedirs(output_dir, exist_ok=True)

for file in file_list:
    df = pd.read_feather(file)
    filtered_volume_bids = df[(df["BIDTYPE"] == "RAISEREG") | (df["BIDTYPE"] == "LOWERREG")]

    # Get just the filename without the path
    base_filename = os.path.basename(file)  # e.g. "PUBLIC_DVD_BIDPEROFFER_D_201201010000.feather"
    # Remove '.feather' extension
    filename_no_ext = os.path.splitext(base_filename)[0]  # e.g. "PUBLIC_DVD_BIDPEROFFER_D_201201010000"

    # Construct the full output path in output_dir
    out_file = os.path.join(output_dir, filename_no_ext + ".parquet")

    # Save to Parquet
    filtered_volume_bids.to_parquet(out_file, index=False)
    print(f"Finished processing {file} -> {out_file}\n")

In [ ]:
# 1. Gather all Parquet files in the directory
all_files = glob.glob('/Volumes/T7/bid-volume-filtered/*.parquet')

# 2. Filter out any hidden dot-underscore files (._filename.parquet)
valid_files = [f for f in all_files if not os.path.basename(f).startswith('._')]

# 3. Print how many valid Parquet files were found
n_files = len(valid_files)
print(f"Found {n_files} valid Parquet file(s) in /Volumes/T7/bid-volume-filtered.\n")

# 4. Iterate over each valid file to show progress
for idx, file_path in enumerate(valid_files, start=1):
    file_name = os.path.basename(file_path)
    print(f"Processing file {idx} of {n_files}: {file_name}")

# 5. Read all valid files into a single Dask DataFrame
print("\nReading all valid Parquet files into a Dask DataFrame...")
ddf = dd.read_parquet(valid_files)
print("Done reading files into Dask DataFrame.\n")

# 7. Display columns and a small sample
print("Columns in the Dask DataFrame:", ddf.columns)
print("\nSample data from the Dask DataFrame:")
print(ddf.head())

Found 933 valid Parquet file(s) in /Volumes/T7/bid-volume-filtered.

Processing file 1 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210401.parquet
Processing file 2 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210402.parquet
Processing file 3 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210403.parquet
Processing file 4 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210404.parquet
Processing file 5 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210405.parquet
Processing file 6 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210406.parquet
Processing file 7 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210407.parquet
Processing file 8 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210408.parquet
Processing file 9 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210409.parquet
Processing file 10 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210410.parquet
Processing file 11 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210411.parquet
Processing file 12 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210412.parquet
Processing file 13 of 933: PUBLIC_DVD_BIDPEROFFER_D_20210413.parquet
Processing file 14 of 933: PUBLIC_DVD_BIDPE

In [25]:
ddf.head()

,SETTLEMENTDATE,DUID,BIDTYPE,OFFERDATE,MAXAVAIL,ENABLEMENTMIN,ENABLEMENTMAX,LOWBREAKPOINT,HIGHBREAKPOINT,BANDAVAIL1,BANDAVAIL2,BANDAVAIL3,BANDAVAIL4,BANDAVAIL5,BANDAVAIL6,BANDAVAIL7,BANDAVAIL8,BANDAVAIL9,BANDAVAIL10,INTERVAL_DATETIME
0,2021/04/01 00:00:00,BALBG1,LOWERREG,2021/03/26 13:30:57,30,0,30,30,30,0,0,0,0,0,0,0,0,0,30,2021/04/01 04:05:00
1,2021/04/01 00:00:00,BALBG1,RAISEREG,2021/03/26 13:30:57,30,0,30,0,0,0,0,0,0,10,10,10,0,0,0,2021/04/01 04:05:00
2,2021/04/01 00:00:00,BALBL1,LOWERREG,2021/03/26 13:38:40,30,0,30,0,0,0,0,0,0,0,0,0,0,0,30,2021/04/01 04:05:00
3,2021/04/01 00:00:00,BALBL1,RAISEREG,2021/03/26 13:38:40,28,0,30,30,30,0,30,0,0,0,0,0,0,0,0,2021/04/01 04:05:00
4,2021/04/01 00:00:00,BARKIPS1,LOWERREG,2021/03/17 01:00:43,0,9,210,126,210,0,0,0,0,0,0,0,0,0,108,2021/04/01 04:05:00


In [26]:
import pandas as pd
import os
import glob
from datetime import datetime, timedelta
import dask.dataframe as dd

def create_settlement_dates_csv(parquet_dir, output_csv_path):
    """
    Creates a CSV file with a row for each day between 2010-2024, indicating
    whether that date exists as a settlement date in the data.
    
    Parameters:
    parquet_dir (str): Directory containing the filtered parquet files
    output_csv_path (str): Path to save the resulting CSV file
    """
    print("Starting settlement dates availability check...")
    
    # Create a dataframe with all dates from 2010 to 2024
    start_date = datetime(2010, 1, 1)
    end_date = datetime(2024, 12, 31)
    all_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    date_df = pd.DataFrame({'date': all_dates})
    # Format date_str to match the format in your data (YYYY/MM/DD)
    date_df['date_str'] = date_df['date'].dt.strftime('%Y/%m/%d')
    # Also create a date_time_str that includes the time component (YYYY/MM/DD HH:MM:SS)
    date_df['date_time_str'] = date_df['date'].dt.strftime('%Y/%m/%d 00:00:00')
    date_df['exists'] = False
    
    # Find all parquet files
    print(f"Looking for parquet files in {parquet_dir}...")
    all_files = glob.glob(os.path.join(parquet_dir, '*.parquet'))
    valid_files = [f for f in all_files if not os.path.basename(f).startswith('._')]
    print(f"Found {len(valid_files)} valid parquet files.")
    
    if not valid_files:
        print(f"No parquet files found in {parquet_dir}. Please check the directory path.")
        return
    
    # Read the settlement dates from parquet files
    print("Reading settlement dates from parquet files...")
    try:
        # Try using Dask for large datasets
        ddf = dd.read_parquet(valid_files)
        
        # Check if 'SETTLEMENTDATE' column exists
        if 'SETTLEMENTDATE' in ddf.columns:
            # Handle the case where SETTLEMENTDATE might be a string column or datetime column
            print("Detected SETTLEMENTDATE column. Checking data type...")
            # Sample a few rows to check the data type
            sample_df = ddf.head(5)
            settlement_date_type = type(sample_df['SETTLEMENTDATE'].iloc[0])
            print(f"SETTLEMENTDATE data type appears to be: {settlement_date_type}")
            
            # If it's already a string with format like "2021/04/01 00:00:00", 
            # we'll parse it to datetime
            # Try to convert to pandas datetime first
            try:
                if ddf['SETTLEMENTDATE'].dtype == 'object':
                    # If it's a string column, parse it to datetime
                    settlement_dates = pd.to_datetime(
                        ddf['SETTLEMENTDATE'].compute(),
                        format='%Y/%m/%d %H:%M:%S'
                    ).dt.floor('D').unique()
                else:
                    # If it's already a datetime column, just use it
                    settlement_dates = ddf['SETTLEMENTDATE'].compute().dt.floor('D').unique()
            except Exception as dt_error:
                print(f"Error converting settlement dates to datetime: {dt_error}")
                # Fallback to treating as strings
                settlement_dates = []
            print(f"Found {len(settlement_dates)} unique settlement dates in the data.")
            
            # Mark dates that exist in the settlement data
            for date in settlement_dates:
                date_str = date.strftime('%Y/%m/%d')
                date_df.loc[date_df['date_str'] == date_str, 'exists'] = True
                
            # Also check for exact datetime string matches (with 00:00:00 time component)
            settlement_datetime_strings = ddf['SETTLEMENTDATE'].astype(str).compute().unique()
            for date_time_str in settlement_datetime_strings:
                if isinstance(date_time_str, str):
                    # For string format like "2021/04/01 00:00:00"
                    date_df.loc[date_df['date_time_str'] == date_time_str, 'exists'] = True
                
        else:
            print("Warning: 'SETTLEMENTDATE' column not found in the parquet files.")
            
    except Exception as e:
        print(f"Error reading parquet files with Dask: {e}")
        print("Falling back to sequential reading...")
        
        # Fallback: read files sequentially with pandas if Dask fails
        all_settlement_dates = set()
        for file in valid_files:
            try:
                df = pd.read_parquet(file)
                if 'SETTLEMENTDATE' in df.columns:
                    # Try to convert to pandas datetime
                    try:
                        if df['SETTLEMENTDATE'].dtype == 'object':
                            # If it's a string, parse with the expected format
                            file_dates = pd.to_datetime(
                                df['SETTLEMENTDATE'], 
                                format='%Y/%m/%d %H:%M:%S'
                            ).dt.floor('D').unique()
                        else:
                            # If it's already datetime
                            file_dates = pd.to_datetime(df['SETTLEMENTDATE']).dt.floor('D').unique()
                    except Exception as dt_error:
                        print(f"Error converting dates in file {file}: {dt_error}")
                        file_dates = []
                    all_settlement_dates.update(file_dates)
            except Exception as file_error:
                print(f"Error reading file {file}: {file_error}")
        
        # Mark dates that exist in the settlement data
        for date in all_settlement_dates:
            date_str = date.strftime('%Y/%m/%d')
            date_df.loc[date_df['date_str'] == date_str, 'exists'] = True
            
        # Also collect and check exact datetime strings
        all_settlement_datetime_strings = set()
        for file in valid_files:
            try:
                df = pd.read_parquet(file)
                if 'SETTLEMENTDATE' in df.columns:
                    # Get the raw string values
                    file_datetime_strings = df['SETTLEMENTDATE'].astype(str).unique()
                    all_settlement_datetime_strings.update(file_datetime_strings)
            except Exception as file_error:
                print(f"Error reading datetime strings from file {file}: {file_error}")
                
        # Mark dates based on exact datetime string matches
        for date_time_str in all_settlement_datetime_strings:
            if isinstance(date_time_str, str):
                # For string format like "2021/04/01 00:00:00"
                date_df.loc[date_df['date_time_str'] == date_time_str, 'exists'] = True
        
        print(f"Found {len(all_settlement_dates)} unique settlement dates in the data.")
    
    # Add some additional columns for convenience
    date_df['year'] = date_df['date'].dt.year
    date_df['month'] = date_df['date'].dt.month
    date_df['day'] = date_df['date'].dt.day
    date_df['weekday'] = date_df['date'].dt.day_name()
    
    # Save to CSV
    print(f"Saving results to {output_csv_path}...")
    date_df.to_csv(output_csv_path, index=False)
    print(f"CSV file created successfully at {output_csv_path}")
    
    # Summary
    exists_count = date_df['exists'].sum()
    total_days = len(date_df)
    print(f"Summary: {exists_count} of {total_days} days have settlement data.")
    
    return date_df

if __name__ == "__main__":
    # You can modify these paths to match your environment
    # For example: if using a local directory instead of an external drive
    parquet_dir = '/Volumes/T7/bid-volume-filtered'  # Change this to your parquet directory
    output_csv_path = 'settlement_dates_availability.csv'  # Output file name
    
    # Alternative path options (uncomment and modify as needed)
    # parquet_dir = './bid-volume-filtered'  # Using a local directory
    # parquet_dir = '/Volumes/T7/bid-price-filtered'  # Using price bid data instead
    
    # Create the CSV file
    result_df = create_settlement_dates_csv(parquet_dir, output_csv_path)
    
    # Print a sample of the results
    if result_df is not None:
        print("\nSample of the created CSV file:")
        print(result_df.head(10))

Starting settlement dates availability check...
Looking for parquet files in /Volumes/T7/bid-volume-filtered...
Found 933 valid parquet files.
Reading settlement dates from parquet files...
Detected SETTLEMENTDATE column. Checking data type...
SETTLEMENTDATE data type appears to be: <class 'str'>


KeyboardInterrupt: 

In [ ]:
import pandas as pd

# Define the file path
file_path = '/Volumes/T7/bid-price-data/PUBLIC_DVD_BIDDAYOFFER_D_20210402.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

# Print the shape (rows, columns)
print(f"DataFrame shape: {df.shape}")

# Show the first few rows
print("\nFirst few rows of the DataFrame:")
print(df.head())

# Show basic info about the DataFrame (column types, non-null counts)
print("\nDataFrame info:")
print(df.info())

# Optionally, describe numeric columns
print("\nDescriptive statistics for numeric columns:")
print(df.describe())

DataFrame shape: (1164, 32)

First few rows of the DataFrame:
   I  BID  BIDDAYOFFER_D  2       SETTLEMENTDATE      DUID     BIDTYPE  \
0  D  BID  BIDDAYOFFER_D  2  2021/04/02 00:00:00     ARWF1      ENERGY   
1  D  BID  BIDDAYOFFER_D  2  2021/04/02 00:00:00   ASNENC1   RAISE6SEC   
2  D  BID  BIDDAYOFFER_D  2  2021/04/02 00:00:00   ASSENC1  RAISE60SEC   
3  D  BID  BIDDAYOFFER_D  2  2021/04/02 00:00:00   ASSENC1   RAISE6SEC   
4  D  BID  BIDDAYOFFER_D  2  2021/04/02 00:00:00  ASTFG1V1  RAISE60SEC   

     BIDSETTLEMENTDATE            OFFERDATE  VERSIONNO  ... PRICEBAND10  \
0  2021/03/30 00:00:00  2021/03/30 12:19:05          1  ...    13025.35   
1  2021/04/02 00:00:00  2021/04/02 03:29:09          1  ...    10000.00   
2  2021/04/02 00:00:00  2021/04/01 18:26:07          1  ...      256.00   
3  2021/04/02 00:00:00  2021/04/01 18:26:07          1  ...      256.00   
4  2021/04/01 00:00:00  2021/04/01 12:24:25          1  ...      256.00   

   MINIMUMLOAD   T1   T2   T3   T4  NORMAL

In [ ]:
# Filter the price bid data 
import os
import glob
import pandas as pd

# Create the output directory if it doesn't already exist
output_dir = '/Volumes/T7/bid-price-filtered'
os.makedirs(output_dir, exist_ok=True)

# List all Feather files in the directory
file_list = glob.glob('/Volumes/T7/bid-price-data/*.feather')

for file in file_list:
    print(f"Processing {file}...")
    # Read the Feather file
    df = pd.read_feather(file)
    
    # Filter to keep rows where BIDTYPE is either "RAISEREG" or "LOWERREG"
    filtered_price_bids = df[(df["BIDTYPE"] == "RAISEREG") | (df["BIDTYPE"] == "LOWERREG")]

    # Print the first few rows of the filtered DataFrame
    print("Filtered DataFrame head:")
    print(filtered_price_bids.head())

    # Construct a new output file path by replacing the directory and file extension
    out_file = file.replace('bid-price-data', 'bid-price-filtered').replace('.feather', '.parquet')

    # Save the filtered DataFrame as a Parquet file
    filtered_price_bids.to_parquet(out_file, index=False)
    print(f"Finished processing {file}.\n")

Processing /Volumes/T7/bid-price-data/PUBLIC_DVD_BIDDAYOFFER_D_20210401.feather...
Filtered DataFrame head:
         SETTLEMENTDATE      DUID   BIDTYPE            OFFERDATE VERSIONNO  \
7   2021/04/01 00:00:00    BALBG1  RAISEREG  2021/03/26 13:30:57         1   
17  2021/04/01 00:00:00  BRAEMAR2  RAISEREG  2020/09/03 15:48:24         1   
18  2021/04/01 00:00:00  BRAEMAR3  LOWERREG  2020/09/03 15:49:58         1   
21  2021/04/01 00:00:00      BW01  RAISEREG  2021/03/31 03:55:20         1   
29  2021/04/01 00:00:00  DEVILS_G  LOWERREG  2021/04/01 01:30:23         1   

   PRICEBAND1 PRICEBAND2 PRICEBAND3 PRICEBAND4 PRICEBAND5 PRICEBAND6  \
7           0       7.89      13.35      28.89      61.89      92.89   
17          0          1          2          4          8         16   
18          0          1          2          4          8         16   
21          1        1.5         11       25.1         50        101   
29       0.01        2.5        3.8          9         14      

KeyboardInterrupt: 

In [ ]:
# import os

# # Define the directory where the sorted files are now located
# sorted_dir = "/Volumes/T7/bid-volume-data-sorted"

# # Iterate through all files in the sorted directory
# for filename in os.listdir(sorted_dir):
#     if filename.startswith("._") or not filename.endswith(".feather"):
#         continue  # Skip hidden macOS files and non-feather files

#     file_path = os.path.join(sorted_dir, filename)

#     # Extract filename without extension
#     name_part, ext = os.path.splitext(filename)  # Splits into name and '.feather'

#     # Debugging info: print filenames being checked
#     print(f"Checking: {filename}")

#     # Check if filename ends with '0000' before '.feather'
#     if name_part.endswith("0000"):
#         new_name_part = name_part[:-4]  # Remove last four zeroes
#         new_filename = new_name_part + ext  # Append '.feather'

#         new_file_path = os.path.join(sorted_dir, new_filename)

#         # Check if the new filename already exists before renaming
#         if os.path.exists(new_file_path):
#             print(f"Skipping {filename}, {new_filename} already exists.")
#             continue

#         # Rename the file
#         os.rename(file_path, new_file_path)
#         print(f"Renamed: {filename} -> {new_filename}")

# print("Filename cleanup complete.")